### 197. Rising Temperature
```
Table: Weather

+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| id            | int     |
| recordDate    | date    |
| temperature   | int     |
+---------------+---------+
id is the column with unique values for this table.
There are no different rows with the same recordDate.
This table contains information about the temperature on a certain day.
 

Write a solution to find all dates' id with higher temperatures compared to its previous dates (yesterday).

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Weather table:
+----+------------+-------------+
| id | recordDate | temperature |
+----+------------+-------------+
| 1  | 2015-01-01 | 10          |
| 2  | 2015-01-02 | 25          |
| 3  | 2015-01-03 | 20          |
| 4  | 2015-01-04 | 30          |
+----+------------+-------------+
Output: 
+----+
| id |
+----+
| 2  |
| 4  |
+----+
Explanation: 
In 2015-01-02, the temperature was higher than the previous day (10 -> 25).
In 2015-01-04, the temperature was higher than the previous day (20 -> 30).
```

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType,StringType,StructType,StructField
from pyspark.sql.functions import col,datediff

In [4]:
spark=SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [7]:
Data=data =[[1, '2015-01-01', 10], [2, '2015-01-02', 25], [3, '2015-01-03', 20], [4, '2015-01-04', 30]]
Schema=StructType([StructField("id",IntegerType(),True),\
                   StructField("recordDate",StringType(),True),\
                   StructField("temperature",IntegerType(),True)])
Weather=spark.createDataFrame(Data,Schema)
Weather.show()

+---+----------+-----------+
| id|recordDate|temperature|
+---+----------+-----------+
|  1|2015-01-01|         10|
|  2|2015-01-02|         25|
|  3|2015-01-03|         20|
|  4|2015-01-04|         30|
+---+----------+-----------+



In [11]:
Weather=Weather.withColumn('recordDate',col('recordDate').cast("date"))
Weather.printSchema()

root
 |-- id: integer (nullable = true)
 |-- recordDate: date (nullable = true)
 |-- temperature: integer (nullable = true)



In [16]:
Weather.alias('W1').join(Weather.alias('W2'),datediff(col("W2.recordDate"),col("W1.recordDate"))==1,'inner')\
    .filter(col("W2.temperature")>col("W1.temperature")).select("W2.id").show()

[Stage 17:=================================================>      (66 + 9) / 75]

+---+
| id|
+---+
|  2|
|  4|
+---+



In [ ]:
### Else try using window functions

In [20]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag,datediff

In [21]:
w = Window.orderBy("recordDate")

result = (
    Weather
    .withColumn("prev_day", lag("recordDate").over(w))
    .withColumn("prev_temp", lag("temperature").over(w))
    .filter(
        (datediff(col("recordDate"), col("prev_day")) == 1) &
        (col("temperature") > col("prev_temp"))
    )
    .select("id")
)

result.show()

+---+
| id|
+---+
|  2|
|  4|
+---+



In [22]:
spark.stop()